# NeoHuman R3GAN — Generate images on CPU

Use this notebook in a **CPU Colab runtime** to generate images from your saved GPU training checkpoint. Run the cells in order. No training or GPU is needed, and the original 45-image ZIP is not needed.

The existing software ZIP and experiment must be in your Google Drive. This notebook reads the original pinned software, verifies the latest checkpoint, and loads only the EMA generator for CPU inference. It writes new images in a separate output subfolder without changing training checkpoints, timers, configuration or writer locks. You do not need to release a stale training lock for this notebook.

If installation asks for a runtime restart, restart and rerun this notebook from the top. CPU inference has been tested locally on Windows using a real CUDA-trained checkpoint; this Colab notebook's cells were syntax/schema checked, but its live Colab installation has not been tested.


## 1 · Connect Drive and select the saved experiment
Keep the ID of the existing experiment. Leave your original training notebook closed while using this separate CPU notebook.


In [ ]:
from pathlib import Path
import sys, subprocess
from google.colab import drive
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/NeoHuman_R3GAN")
EXPERIMENT_ID = "neohuman-colab-001"
CODE_ROOT = Path("/content/NeoHuman_R3GAN_cpu_code")
CODE_BUNDLE = PROJECT / "software" / "NeoHuman_R3GAN_Workspace.zip"
if not (PROJECT / "experiments" / EXPERIMENT_ID / "config.json").is_file():
    raise FileNotFoundError("Saved experiment not found. Check your Google account, project path and experiment ID.")
if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("The pinned software requires Python 3.11–3.13.")


## 2 · Load the original software and CPU dependencies
Uses the software ZIP already stored on Drive. If missing, upload the original software ZIP when prompted. This does not alter that ZIP or your training notebook.


In [ ]:
import hashlib, importlib, json, os, zipfile
from pathlib import PurePosixPath
if not CODE_BUNDLE.exists():
    from google.colab import files
    print("Upload NeoHuman_R3GAN_Workspace.zip (the SOFTWARE package)")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one workspace ZIP")
    payload = next(iter(uploaded.values()))
    CODE_BUNDLE.parent.mkdir(parents=True, exist_ok=True)
    temporary = CODE_BUNDLE.with_suffix(".partial")
    temporary.write_bytes(payload)
    os.replace(temporary, CODE_BUNDLE)
with zipfile.ZipFile(CODE_BUNDLE) as package:
    inventory = json.loads(package.read("workspace-package.json"))["files"]
    if set(package.namelist()) != set(inventory) | {"workspace-package.json"}:
        raise ValueError("Workspace ZIP inventory mismatch")
    for name, expected in inventory.items():
        p = PurePosixPath(name)
        if p.is_absolute() or ".." in p.parts or "\\" in name or ":" in name:
            raise ValueError("Unsafe package path")
        data = package.read(name)
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError("Workspace ZIP checksum mismatch")
        target = CODE_ROOT / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
loaded = {name: str(getattr(sys.modules[name], "__version__", "")) for name in ("torch", "numpy", "PIL") if name in sys.modules}
subprocess.run([sys.executable, "-m", "pip", "install", "pip==25.1.1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.7.1", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "scripts/fetch_upstream.py"], check=True)
for name, pin in {"torch": "2.7.1", "numpy": "2.2.6", "PIL": "11.2.1"}.items():
    if name in loaded and loaded[name].split("+")[0] != pin:
        raise RuntimeError("Dependencies changed. Restart the Colab session, then rerun this CPU notebook from the top.")
importlib.invalidate_caches()


## 3 · Load the CPU image-generation helper
This cell defines the function; the next cell creates your images.


In [ ]:
"""Read a verified CUDA training checkpoint and generate EMA images on CPU only.

Kept outside neo_r3gan so existing training checkpoints retain their code identity.
No training, optimizer restoration, CUDA tensor allocation, or writer-lock changes.
"""
from pathlib import Path
import tempfile
import uuid

import torch
from PIL import Image
from neo_r3gan.checkpoints import Checkpoints
from neo_r3gan.config import config_hash, load_config
from neo_r3gan.data import grid
from neo_r3gan.io_utils import atomic_write, verified_copy, write_json
from neo_r3gan.upstream import code_version, networks_and_loss, require_pins


def generate_cpu(project, experiment_id, seeds=range(4)):
    seeds = list(seeds)
    if not seeds or len(seeds) > 256 or any(type(s) is not int or not 0 <= s < 2**63 for s in seeds):
        raise ValueError("Use 1..256 integer seeds, each in 0..2**63-1")
    if len(seeds) != len(set(seeds)):
        raise ValueError("Use distinct seeds for distinct output filenames")
    project = Path(project)
    if Path(experiment_id).name != experiment_id or experiment_id in ("", ".", "..") or "\\" in experiment_id:
        raise ValueError("Use the experiment folder name, not a path")
    run = project / "experiments" / experiment_id
    config = load_config(run / "config.json")
    if config["experiment_id"] != experiment_id:
        raise ValueError("Saved configuration belongs to a different experiment")
    require_pins()
    checkpoint, entry = Checkpoints(run, run).latest()
    # Copy before loading so subsequent retention by another runtime cannot
    # remove the file we are reading. No training files are written here.
    with tempfile.TemporaryDirectory(prefix="neohuman-cpu-") as temporary:
        snapshot = Path(temporary) / checkpoint.name
        if verified_copy(checkpoint, snapshot) != entry["sha256"]:
            raise ValueError("Checkpoint changed while copying; retry")
        state = torch.load(snapshot, map_location="cpu", weights_only=True)
    if state["schema"] != 1 or state["code"] != code_version() or state["config_sha256"] != config_hash(config) or state["config"] != config:
        raise ValueError("Checkpoint/config/software mismatch; use the original software package")
    for key in ("step", "training_seconds"):
        if state["counters"][key] != entry[key]:
            raise ValueError("Checkpoint counters differ from its manifest")
    Generator, _, _ = networks_and_loss()
    model = config["model"]
    # Preserve the notebook's CPU random stream; no CUDA RNG is inspected.
    with torch.random.fork_rng(devices=[]):
        generator = Generator(
            NoiseDimension=model["noise_dim"], WidthPerStage=model["widths"],
            BlocksPerStage=model["blocks"], CardinalityPerStage=model["cardinalities"],
            ExpansionFactor=model["expansion"],
        ).to("cpu").eval().requires_grad_(False)
    generator.load_state_dict(state["G_ema"], strict=True)
    del state
    output = run / "generated" / f"cpu-step-{entry['step']:09d}-{uuid.uuid4().hex[:8]}"
    output.mkdir(parents=True, exist_ok=False)
    print(f"Loaded step {entry['step']} / {entry['training_seconds']/3600:.4f} saved training hours. Generating on CPU.", flush=True)
    paths, images = [], []
    with torch.inference_mode():
        for index, seed in enumerate(seeds, 1):
            noise = torch.randn(1, model["noise_dim"], generator=torch.Generator(device="cpu").manual_seed(seed), device="cpu")
            pixels = ((generator(noise)[0].float() + 1) * 127.5).round().clamp(0, 255).byte()
            image = Image.fromarray(pixels.permute(1, 2, 0).numpy())
            path = output / f"seed-{seed:06d}.png"
            atomic_write(path, lambda f: image.save(f, format="PNG"))
            paths.append(path)
            images.append(image)
            print(f"Saved {index}/{len(seeds)}: {path.name}", flush=True)
    preview = output / "preview.png"
    canvas = grid(images, columns=min(4, len(images)))
    atomic_write(preview, lambda f: canvas.save(f, format="PNG"))
    write_json(output / "generation.json", {"device": "cpu", "torch": str(torch.__version__), "seeds": seeds, "checkpoint": entry, "code": code_version()})
    print("Images saved to:", output, flush=True)
    return {"paths": paths, "preview": preview, "output": output, "checkpoint": entry}


## 4 · Generate and save images
Start with four images. Change `START_SEED` to get another set. The same checkpoint and seeds reproduce the same images on the same CPU/software setup; CPU and GPU outputs need not be bitwise identical. Each run saves to a new folder.


In [ ]:
from IPython.display import display
START_SEED = 0
NUM_IMAGES = 4
torch.set_num_threads(2)
RESULT = generate_cpu(PROJECT, EXPERIMENT_ID, seeds=range(START_SEED, START_SEED + NUM_IMAGES))
display(Image.open(RESULT["preview"]))
print("Open this folder in Google Drive:", RESULT["output"])
